# Tracking de experimentos y diagnóstico de entrenamiento

Tres flujos: (a) un run de entrenamiento con auditoría de gradientes por canal,
(b) un study de hiperparámetros con objetivo compuesto y poda, y (c) seguir o
reanudar un study desde disco. Todo queda en `.soma/` para que un front lo lea.


In [ ]:
import json, pathlib
import torch
import torch.nn as nn

import soma
from soma import ChannelConfig, DifferentiableFilter, Graph, search


## (a) Run de entrenamiento con auditoría

`track_run` crea `.soma/runs/<run_id>/` (manifest con git/host, status con
heartbeat, topología del grafo, events/metrics.jsonl). `gradient_audit` con
`channels=` añade diagnósticos por canal: canales muertos, dormidos (Sokar
2023), ignorados (gradient starvation) y leakage entre grupos (CKA).


In [ ]:
class Encoder(DifferentiableFilter):
    lr: float = search(1e-3, 1e-1, scale="log")

    def build_module(self, input_shape):
        return nn.Sequential(nn.Linear(input_shape[-1], 16), nn.ReLU(), nn.Linear(16, 8))

    def output_shape(self, input_shape):
        return (*input_shape[:-1], 8)

g = Graph()
g.node("encoder", Encoder())
x = torch.randn(64, 12)
y = torch.randn(64, 8)
g.materialize(x)
g.train()
g.make_optimizer(lr=0.01)


In [ ]:
with g.track_run("baseline", tags=["demo"]) as run:
    cfg = ChannelConfig(snapshot_every=10,
                        groups={"encoder": {"a": range(0, 4), "b": range(4, 8)}})
    with g.gradient_audit(channels=cfg) as audit:
        module = dict(g.filters())["encoder"]._module
        for epoch in range(5):
            run.log_epoch(epoch, total=5)
            with g.context() as ctx:
                g.zero_grad()
                loss = ((module(x) - y) ** 2).mean()
                g.backward(ctx, loss)   # snapshot del audit + StepCompleted
            g.step(ctx)
            run.log("loss", float(loss), step=epoch)
    print(audit.report().pretty())

run_dir = pathlib.Path(run.dir)
print(sorted(p.name for p in run_dir.iterdir()))


In [ ]:
# Todo lo que un front necesita: eventos, métricas y diagnósticos
print((run_dir / "graph.mmd").read_text())
print(json.loads((run_dir / "status.json").read_text()))
print((run_dir / "diagnostics" / "report.json").read_text()[:400])


## (b) Study con grid, objetivo compuesto y poda

El espacio sale de los descriptores `search()` de los filtros
(`graph.search_space()`, nombres `nodo.param`). El objetivo puede ser un
callable sobre las métricas; la poda usa la regla de la mediana vía
`trial.report()`.


In [ ]:
study = g.study("demo-grid", strategy="grid", n_trials=3,
                objective=lambda m: m["fit"] - 0.1 * m["cost"],
                direction="maximize", pruning=("median", 2))

def train(trial):
    g.apply_params(trial.params)
    lr = trial["encoder.lr"]
    for step in range(8):
        fit = 1.0 - abs(lr - 0.01) * 10 + step * 0.01
        if trial.report("fit", fit, step):
            return None                      # podado
    return {"fit": fit, "cost": lr * 100}

study.run(train, on_event=lambda e: print(" →", e["event_type"]))
print(study.best_trial)
print("run dir:", study.run_dir)


## (c) Seguir y reanudar desde disco

`study.json` se reescribe atómicamente tras cada trial: desde cualquier
máquina con acceso al directorio se puede cargar el estado, y `resume=True`
continúa exactamente donde quedó (sin repetir puntos del grid).


In [ ]:
reloaded = soma.Study.load(study.run_dir)
print(reloaded.progress, len(reloaded.trials))
# reloaded.run(train, resume=True)   # continuaría si quedaran trials

for exp in soma.experiments():
    print(exp["name"], exp["metrics"], exp["tags"])
